# Module 02-1: 첫 번째 LangGraph - 2노드 인사 그래프 (솔루션)

## 학습 목표

1. **TypedDict로 상태 정의**: LangGraph 그래프의 상태(State)를 TypedDict로 설계한다
2. **노드 함수 구현**: 상태를 입력받아 변환된 상태를 반환하는 노드 함수를 작성한다
3. **StateGraph 조립**: `StateGraph`에 노드와 엣지를 추가하여 그래프를 완성한다
4. **그래프 컴파일 및 실행**: `compile()` 후 `invoke()`로 그래프를 실행한다
5. **상태 흐름 이해**: 노드 간 상태가 어떻게 전달되고 변환되는지 파악한다

---
> 참고 문서: `docs/part1-foundations/02-langgraph-fundamentals.md`

## LangGraph를 사용하는 이유

직접 구현하는 방식과 LangGraph를 사용하는 방식을 비교하면 다음과 같습니다.

| 직접 구현 | LangGraph 사용 |
|----------|---------------|
| 상태 관리를 직접 코딩 | TypedDict로 간단히 정의 |
| 분기 로직을 if-else로 복잡하게 | 조건부 엣지로 깔끔하게 |
| 실행 흐름 추적이 어려움 | Mermaid 자동 시각화 |
| 에러 시 어디서 멈췄는지 불명 | 체크포인트로 자동 추적 |
| 테스트가 어려움 | 노드 단위 테스트 가능 |

In [ ]:
# 환경 설정
import sys
sys.path.insert(0, '..')

from typing import TypedDict
from langgraph.graph import StateGraph, START, END

print("환경 설정 완료")

## 실습 1: GreetingState 정의 (솔루션)

In [ ]:
# 솔루션
class GreetingState(TypedDict):
    name: str
    greeting: str
    formatted_output: str

print("GreetingState 정의 완료")

In [ ]:
# 검증 셀
assert 'GreetingState' in dir(), "GreetingState가 정의되어야 합니다"
sample_state: GreetingState = {"name": "테스트", "greeting": "", "formatted_output": ""}
assert sample_state["name"] == "테스트", "name 필드가 올바르게 동작해야 합니다"
print("✅ 실습 1 완료!")

## 실습 2: 노드 함수 구현 (솔루션)

In [ ]:
# 솔루션
def generate_greeting(state: GreetingState) -> dict:
    """이름으로 인사말을 생성하는 노드"""
    return {"greeting": f"안녕하세요, {state['name']}님!"}


def format_output(state: GreetingState) -> dict:
    """인사말을 포맷팅하는 노드"""
    return {"formatted_output": f"[인사말] {state['greeting']}"}


print("노드 함수 구현 완료")

In [ ]:
# 검증 셀
test_state = {"name": "LangGraph", "greeting": "", "formatted_output": ""}
greeting_result = generate_greeting(test_state)
assert "greeting" in greeting_result, "generate_greeting은 'greeting' 키를 반환해야 합니다"
assert "LangGraph" in greeting_result["greeting"], "greeting에 이름이 포함되어야 합니다"

test_state["greeting"] = greeting_result["greeting"]
format_result = format_output(test_state)
assert "formatted_output" in format_result, "format_output은 'formatted_output' 키를 반환해야 합니다"
assert "[인사말]" in format_result["formatted_output"], "formatted_output에 '[인사말]' 접두사가 있어야 합니다"
print("✅ 실습 2 완료!")

## 실습 3: StateGraph 조립 및 실행 (솔루션)

In [ ]:
# 솔루션
graph = StateGraph(GreetingState)

# 노드 추가
graph.add_node("generate_greeting", generate_greeting)
graph.add_node("format_output", format_output)

# 엣지 연결
graph.add_edge(START, "generate_greeting")
graph.add_edge("generate_greeting", "format_output")
graph.add_edge("format_output", END)

# 컴파일
app = graph.compile()

print("그래프 조립 완료")

In [ ]:
# 검증 셀
assert 'app' in dir(), "app 변수(컴파일된 그래프)가 정의되어야 합니다"
result = app.invoke({"name": "학습자", "greeting": "", "formatted_output": ""})
assert "greeting" in result, "결과에 greeting 필드가 있어야 합니다"
assert "formatted_output" in result, "결과에 formatted_output 필드가 있어야 합니다"
assert "학습자" in result["greeting"], "greeting에 이름이 포함되어야 합니다"
assert "[인사말]" in result["formatted_output"], "formatted_output에 '[인사말]' 접두사가 있어야 합니다"
print(f"실행 결과: {result['formatted_output']}")
print("✅ 실습 3 완료!")

## 실습 4: 다양한 입력으로 테스트 (솔루션)

In [ ]:
# 솔루션
test_names = ["Alice", "Bob", "LangGraph"]

for name in test_names:
    result = app.invoke({"name": name, "greeting": "", "formatted_output": ""})
    print(result["formatted_output"])

In [ ]:
# 검증 셀
for name in test_names:
    r = app.invoke({"name": name, "greeting": "", "formatted_output": ""})
    assert name in r["greeting"], f"{name}에 대한 greeting이 올바르지 않습니다"
    assert "[인사말]" in r["formatted_output"], f"{name}에 대한 formatted_output이 올바르지 않습니다"
print("✅ 실습 4 완료!")

## 정리

### 오늘 배운 내용
- `TypedDict`로 그래프의 **상태(State)**를 정의하는 방법
- 상태를 변환하는 **노드 함수** 작성 패턴
- `StateGraph`로 노드와 엣지를 연결하여 **그래프 조립**
- `compile()` 후 `invoke()`로 그래프 **실행**